In [1]:
# Prefilled. Just copy and execute.
import os, math, re, random, zipfile
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models


In [ ]:

# Voir le répertoire courant
print("Répertoire courant :", os.getcwd())

# Lister les fichiers disponibles
print("\nContenu du dossier courant :")
for f in os.listdir('.'):
    print(f)

print("\nContenu du dossier 'data' (si existe) :")
if os.path.exists('data'):
    for f in os.listdir('data'):
        print(f)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Chercher le fichier ZIP sur ton Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.zip'):
            print(os.path.join(root, f))

/content/drive/MyDrive/My website/fahimcoulibaly.zip
/content/drive/MyDrive/React/react-sample.zip
/content/drive/MyDrive/Elementor/elementor-pro.zip
/content/drive/MyDrive/Elementor/pro-elements (2).zip
/content/drive/MyDrive/dataset/Dogs vs Cats.zip


In [ ]:
np.random.seed(42); tf.random.set_seed(42)

# Paths - change if needed
DATA_ROOT = Path("data")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# Build DataFrames from folders
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat","cats"}:
                label = "cat"
            elif parent in {"dog","dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Train validation split
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)
# Unlabeled test for inference only
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

print({"train": train_flow.samples, "val": val_flow.samples, "test": test_flow.samples,
       "class_indices": train_flow.class_indices})

In [ ]:
print(df_tr['label'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Get a batch of training images and labels
images, labels = next(train_flow)

# Get class names from the generator's class_indices
class_names = list(train_flow.class_indices.keys())

plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    # Convert numerical label back to class name
    predicted_label = class_names[int(labels[i])]
    plt.title(predicted_label)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Architecture du CNN pour la Classification Chat vs Chien

Ce réseau de neurones convolutifs (CNN) sera conçu pour distinguer les images de chats de celles de chiens. Voici les détails de l'architecture :

1.  **Blocs Convolutionnels :** Nous utiliserons deux blocs convolutionnels. Chaque bloc commencera par une couche `Conv2D` avec une taille de filtre de (3,3) et une fonction d'activation `relu`.
    *   Le premier bloc utilisera 32 filtres et définira la forme d'entrée comme (180, 180, 3) pour correspondre à la taille de nos images. Il sera suivi d'une couche `MaxPooling2D`.
    *   Le deuxième bloc utilisera 64 filtres (une pratique courante est d'augmenter le nombre de filtres après chaque couche de pooling) et sera également suivi d'une couche `MaxPooling2D`. Le *MaxPooling* est crucial pour réduire les dimensions spatiales des cartes de caractéristiques, ce qui diminue le nombre de paramètres et rend le modèle plus robuste aux petites translations dans l'image.

2.  **Aplatissement (Flattening) :** Après les blocs convolutionnels et de pooling, la sortie multidimensionnelle est aplatie en un vecteur 1D via une couche `Flatten`. Cela prépare les données pour les couches denses (fully connected).

3.  **Couches Denses (Fully Connected) :**
    *   Une couche `Dense` cachée avec 128 neurones et une fonction d'activation `relu` sera ajoutée pour apprendre des combinaisons de caractéristiques de haut niveau.
    *   **Dropout :** Une couche `Dropout` avec un taux de 0.5 sera insérée après la première couche dense. Le *Dropout* est une technique de régularisation efficace qui désactive aléatoirement une fraction des neurones pendant l'entraînement. Cela empêche le modèle de trop s'adapter aux données d'entraînement (surapprentissage) et améliore sa capacité à généraliser sur de nouvelles données.
    *   **Couche de Sortie :** La couche finale sera une couche `Dense` avec un seul neurone (`units=1`) et une fonction d'activation `sigmoid`. Pour une tâche de classification binaire (deux classes : chat ou chien), la fonction d'activation `sigmoid` est appropriée car elle produit une probabilité entre 0 et 1. Une valeur proche de 0 indiquera une classe (par exemple, chat) et une valeur proche de 1 indiquera l'autre classe (par exemple, chien). La *binary cross-entropy* sera la fonction de perte utilisée lors de la compilation du modèle, car elle est correcte pour modéliser une cible de Bernoulli avec une sortie sigmoïde.

### Pourquoi cette architecture?

*   **Clarté de l'hypothèse et biais inductif:** La définition explicite de l'architecture clarifie les capacités d'apprentissage du modèle.
*   **Convolution (Conv2D):** Efficace pour capter les motifs locaux dans les images, comme les bords et les textures.
*   **Pooling (MaxPooling2D):** Confère une invariance translationnelle au modèle et réduit le nombre de paramètres, ce qui aide à la généralisation et diminue les coûts de calcul.

### Point clé d'apprentissage:

*   **Binary Cross-Entropy:** C'est la fonction de perte appropriée pour modéliser une cible de Bernoulli (classification binaire) lorsqu'elle est associée à une fonction d'activation sigmoïde dans la couche de sortie. La fonction Softmax, quant à elle, est utilisée pour la classification multi-classes (plus de deux classes).

In [ ]:
model = models.Sequential([
    # Premier bloc convolutionnel
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Deuxième bloc convolutionnel
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Aplatissement
    layers.Flatten(),

    # Couche dense cachée
    layers.Dense(128, activation='relu'),

    # Couche Dropout pour la régularisation
    layers.Dropout(0.5),

    # Couche de sortie pour la classification binaire
    layers.Dense(1, activation='sigmoid')
])

# Afficher un résumé du modèle
model.summary()

## 4. Choix de la configuration d'optimisation

Pour entraîner efficacement notre modèle, il est essentiel de bien choisir les hyperparamètres d'optimisation :

*   **Optimiseur :** Nous utiliserons l'optimiseur **Adam**. Il est recommandé pour sa rapidité de convergence et sa capacité à s'adapter aux différents gradients des paramètres.
*   **Taux d'apprentissage initial :** Nous définirons un taux d'apprentissage initial raisonnable qui permettra au modèle de commencer l'apprentissage sans divergence ni progression trop lente.
*   **Taille du lot (Batch size) :** La taille du lot est choisie en fonction de la mémoire disponible (GPU ou CPU) et de la taille du dataset. Une taille de lot de 32 a été choisie, offrant un bon compromis entre la vitesse d'entraînement et la stabilité du gradient.
*   **`EarlyStopping` :** Cette technique est cruciale pour prévenir le surapprentissage. L'entraînement sera arrêté si la perte de validation (`val_loss`) ne s'améliore pas après un certain nombre d'époques (patience).
*   **`ReduceLROnPlateau` (facultatif) :** Peut être utilisé pour adapter le taux d'apprentissage en le réduisant lorsque la performance de validation stagne, aidant ainsi le modèle à sortir des plateaux de convergence.

### Justification :

Ces éléments permettent de contrôler la dynamique de l'entraînement. L'arrêt anticipé (`EarlyStopping`) évite le surapprentissage en stoppant l'entraînement lorsque la performance sur l'ensemble de validation commence à se dégrader. La planification du taux d'apprentissage (`ReduceLROnPlateau`) aide le modèle à s'échapper des minima locaux et à converger vers une meilleure solution.

### Point clé d'apprentissage :

Il est important de surveiller à la fois la perte (`loss`) et la précision (`accuracy`) pendant l'entraînement. La précision peut être trompeuse en cas de déséquilibre des classes, tandis que la perte est une mesure plus lisse et plus sensible à la qualité des probabilités prédites par le modèle.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5,          # Stop after 5 epochs if val_loss doesn't improve
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity.
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor validation loss
    factor=0.2,          # Reduce learning rate by 20%
    patience=3,          # If val_loss doesn't improve for 3 epochs
    min_lr=0.00001,      # Minimum learning rate
    verbose=1
)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(train_flow, validation_data=val_flow, epochs=10)

In [ ]:
# Compile the model with the Adam optimizer, binary cross-entropy loss, and accuracy as a metric
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model with the defined callbacks
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=30, # Increased epochs as EarlyStopping will handle termination
    callbacks=[early_stopping, reduce_lr]
)

Let's visualize the training and validation accuracy and loss over epochs. This helps us understand if the model is overfitting.

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
def plot_training_history(history):
    plt.figure(figsize=(12, 5))

    # Plot training & validation accuracy values
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    # Plot training & validation loss values
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.show()

plot_training_history(history)

The `test_flow` was created with `class_mode=None` because `df_test_full` was built with `labeled=False`. This means it does not contain true labels for evaluation. Thus, `model.evaluate(test_flow)` will not work directly. Instead, we can generate predictions on this unlabeled test set.

In [ ]:
# Generate predictions for the unlabeled test set
predictions = model.predict(test_flow)

# Display the first 10 predictions
print("First 10 predictions (probabilities for 'dog' class):")
print(predictions[:10].flatten())

# To get binary labels (0 for cat, 1 for dog) based on a threshold (e.g., 0.5):
predicted_labels = (predictions > 0.5).astype(int)
print("\nFirst 10 predicted labels (0=cat, 1=dog):")
print(predicted_labels[:10].flatten())

# If you had true labels for the test set, you could convert predictions to class names:
# class_names = list(train_flow.class_indices.keys())
# predicted_class_names = [class_names[label] for label in predicted_labels.flatten()]
# print(f"\nFirst 10 predicted class names: {predicted_class_names[:10]}")

In [ ]:
# test_loss, test_accuracy = model.evaluate(test_flow)
# print(f"Test accuracy: {test_accuracy: .4f}")